<a href="https://colab.research.google.com/github/ymuto0302/PJ2025/blob/main/Autoencoder_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MNIST を題材とした自己符号化器（Autoencoder, AE）の PyTorch 実装

ネットワーク構造:
```
エンコーダ:  784 -> 256 -> 64 -> 32   （入力を段階的に絞り込む）
デコーダ:    32 -> 64 -> 256 -> 784    （エンコーダの鏡像構造）
```

入力画像（28x28 = 784 次元）を 32 次元の潜在表現へ圧縮し，そこから元の画像を復元するよう学習する。学習には正解ラベルを用いず，入力データそのものを復元の目標とするため，教師なし学習に分類される。

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import matplotlib.pyplot as plt

In [15]:
# ---------------------------------------------------------------------------
# モデル定義
# ---------------------------------------------------------------------------
class Autoencoder(nn.Module):
    """全結合層のみで構成した自己符号化器である．

    エンコーダとデコーダはユニット数が左右対称になるよう設計しており，
    中央の latent_dim（既定 32）が砂時計の最も細い「くびれ」にあたる．
    """

    def __init__(self, input_dim: int = 784, latent_dim: int = 32):
        super().__init__()

        # エンコーダ: 784 -> 256 -> 64 -> 32
        # 潜在表現を出力する最終層には活性化を置かず，
        # 値域を制約しない線形出力としている。
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, latent_dim),
        )

        # デコーダ: 32 -> 64 -> 256 -> 784
        # 最終層の後の Sigmoid で出力を [0, 1] に収め，
        # [0, 1] に正規化した入力画素と比較できるようにしている。
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """入力から潜在表現（32 次元）を取り出す．特徴抽出に用いる．"""
        return self.encoder(x)

    def decode(self, z):
        """潜在表現から画像を復元する．"""
        return self.decoder(z)

    def forward(self, x):
        """入力 x を圧縮し，復元したテンソルを返す．"""
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [16]:
# ---------------------------------------------------------------------------
# データ
# ---------------------------------------------------------------------------
def build_dataloaders(data_root, batch_size):
    """MNIST の学習用・テスト用 DataLoader を構築する。

    ToTensor() により画素値は自動的に [0, 1] へ正規化される。画像は
    後段で 784 次元へ平坦化（flatten）して全結合層に入力する。
    """
    transform = transforms.ToTensor()

    train_set = datasets.MNIST(
        root=data_root, train=True, download=True, transform=transform
    )
    test_set = datasets.MNIST(
        root=data_root, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=2
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False, num_workers=2
    )
    return train_loader, test_loader


In [17]:
# ---------------------------------------------------------------------------
# 学習・評価
# ---------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device):
    """1 エポック分の学習を行い，平均損失を返す"""
    model.train()
    running_loss = 0.0
    n_samples = 0

    for images, _ in loader:  # ラベルは使わない（教師なし学習）
        images = images.to(device)
        # (N, 1, 28, 28) -> (N, 784) へ平坦化する
        x = images.view(images.size(0), -1)

        x_hat = model(x)
        # 復元誤差を計算する．入力 x 自身が復元の目標（お手本）である
        loss = criterion(x_hat, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    return running_loss / n_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """テストデータでの平均復元誤差を返す．"""
    model.eval()
    running_loss = 0.0
    n_samples = 0

    for images, _ in loader:
        images = images.to(device)
        x = images.view(images.size(0), -1)
        x_hat = model(x)
        loss = criterion(x_hat, x)
        running_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    return running_loss / n_samples


In [18]:
# ---------------------------------------------------------------------------
# 可視化
# ---------------------------------------------------------------------------
@torch.no_grad()
def save_reconstructions(model, loader, device, out_path, n=10):
    """テスト画像（上段）と，その復元結果（下段）を並べて保存する．"""
    model.eval()
    images, _ = next(iter(loader))
    images = images[:n].to(device)
    x = images.view(images.size(0), -1)
    x_hat = model(x)

    # 表示用に (N, 28, 28) へ戻す
    originals = x.view(-1, 28, 28).cpu().numpy()
    reconstructions = x_hat.view(-1, 28, 28).cpu().numpy()

    fig, axes = plt.subplots(2, n, figsize=(n * 1.2, 2.6))
    for i in range(n):
        axes[0, i].imshow(originals[i], cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i], cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_ylabel("input", rotation=0, labelpad=30)
    axes[1, 0].set_ylabel("output", rotation=0, labelpad=30)
    fig.suptitle("Top: input   /   Bottom: reconstruction")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"再構成結果を {out_path} に保存した．")


In [19]:
# ---------------------------------------------------------------------------
# メイン
# ---------------------------------------------------------------------------
def main():
    EPOCHS = 20 # 学習エポック数
    BATCH_SIZE = 256 # バッチサイズ
    LR = 1e-3 # 学習率
    LATENT_DIM = 32 # 潜在表現の次元数

    DATA_ROOT = "./data" # データ保存先
    OUT_DIR = "./outputs" # 成果物の保存先

    # 保存用ディレクトリの作成
    os.makedirs(OUT_DIR, exist_ok=True)

    # 利用可能なら GPU を使う
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # データ
    train_loader, test_loader = build_dataloaders(DATA_ROOT, BATCH_SIZE)

    # モデル
    model = Autoencoder(input_dim=784, latent_dim=LATENT_DIM).to(device)
    print(model)

    # 損失関数 (MSE) : 画素値を連続量とみなし二乗誤差で測る
    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # 学習ループ
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        test_loss = evaluate(model, test_loader, criterion, device)
        print(
            f"[epoch {epoch:3d}/{EPOCHS}] "
            f"train_loss={train_loss:.6f}  test_loss={test_loss:.6f}"
        )

    # 学習済みモデルの保存
    ckpt_path = os.path.join(OUT_DIR, "mnist_ae.pth")
    torch.save(model.state_dict(), ckpt_path)
    print(f"モデルを {ckpt_path} に保存した．")

    # 再構成結果の可視化
    save_reconstructions(
        model, test_loader, device, os.path.join(OUT_DIR, "reconstruction.png")
    )

    # 潜在表現の形を確認する（特徴抽出・異常検知への入り口）
    '''
    sample_images, _ = next(iter(test_loader))
    sample_x = sample_images.view(sample_images.size(0), -1).to(device)
    with torch.no_grad():
        z = model.encode(sample_x)
    print(f"潜在表現の形状: {tuple(z.shape)}  （784 次元から圧縮された）")
    '''


if __name__ == "__main__":
    main()

使用デバイス: cuda
Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=64, out_features=256, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=256, out_features=784, bias=True)
    (5): Sigmoid()
  )
)
[epoch   1/20] train_loss=0.069851  test_loss=0.044350
[epoch   2/20] train_loss=0.035223  test_loss=0.029488
[epoch   3/20] train_loss=0.026919  test_loss=0.023938
[epoch   4/20] train_loss=0.022699  test_loss=0.020899
[epoch   5/20] train_loss=0.020230  test_loss=0.019125
[epoch   6/20] train_loss=0.018544  test_loss=0.017528
[epoch   7/20] train_loss=0.017060  test_loss=0.016282
[epoch   8/20] train_loss=0.0160